# Script 14 - Figure 1, the distribution supplement, and sample accounting

## Figure 1 - study overview (main)

- **Panel a**: conceptual framework. The exposure node is the rest-activity
  rhythm construct (IV, IS, RA).
- **Panel b**: sample-selection flow, terminating in the two primary SEM
  analysis sets. Every count is derived from the analytic file and
  cross-checked against the n reported in the Mplus output files.

Written as `figure/main/Fig1_Study_Overview.pdf` (vector) + 600 dpi PNG, with
`figure/source_data/Source_Data_Fig1_Study_Overview.xlsx`.

## Supp Fig S4 - exposure and mediator distributions

Five panels: the three rhythm metrics as stratified histograms, the two
mediators as box plots, on the complete-case depression sample. Written as
`figure/supplementary/SuppFigS4_Exposure_Mediator_Distributions.{pdf,png}`
with `figure/source_data/Source_Data_SuppFigS4_Exposure_Mediator_Distributions.xlsx`.

## Supp Table S12 - sample accounting

One row per analysis set used in the manuscript, each with its inclusion
rule, its n, and where it is reported. The accelerometry step is counted
separately for IV/IS (hourly profile) and RA (minute-level M10/L5). Written as
`result/supplementary/SuppTableS12_Sample_Accounting.{csv,xlsx}`.

Reads `data/processed/8_nhanes_analytic.csv` and the sample sizes in
`scripts/9_mplus_models/*.out`.


## Step 0 — Imports and paths

In [ ]:
import os, sys, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
from matplotlib.gridspec import GridSpec
warnings.filterwarnings('ignore')

def find_root():
    """Locate the project root without assuming where the kernel started."""
    env = os.environ.get('NHANES_SEM_ROOT')
    cands = ([Path(env)] if env else []) + [Path.cwd(), *Path.cwd().parents]
    for q in cands:
        if (q / 'data' / 'processed').is_dir() and (q / 'scripts').is_dir():
            return q
    raise RuntimeError(
        'cannot find the project root from ' + str(Path.cwd())
        + ' -- start Jupyter inside the project, or set NHANES_SEM_ROOT')

BASE   = find_root()
PROC   = BASE / 'data' / 'processed'
F_MAIN = BASE / 'figure' / 'main'
F_SUPP = BASE / 'figure' / 'supplementary'
F_SRC  = BASE / 'figure' / 'source_data'
R_SUPP = BASE / 'result' / 'supplementary'
for q in (F_MAIN, F_SUPP, F_SRC, R_SUPP):
    q.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(BASE / 'scripts'))

# Reload project modules so the notebook runs against the code on disk.
import importlib
import nature_palette
for _m in (nature_palette,):
    importlib.reload(_m)

from nature_palette import (KDM_BA_COLOR, SII_COLOR, PHQ9_COLOR, COG_COLOR,
                            IV_GRADIENT_4, GREY, BLACK)

from nature_palette import EXP_COL, EXP_ORDER  # one source of truth
EXP_NAME = {'IV': 'Intradaily variability', 'IS': 'Interdaily stability',
            'RA': 'Relative amplitude'}
IV_COLOR = EXP_COL['IV']


## Step 1 — Load data and derive the canonical variables

The cognition composite is the mean of the within-60+ z-scores of the memory,
fluency and processing-speed domain scores, with raw zeros recoded to missing,
as in `scripts/9_mplus_prep.ipynb`.

In [ ]:
df = pd.read_csv(PROC / '8_nhanes_analytic.csv', low_memory=False)
assert 'WTMEC4YR' in df.columns, 'run script 8: WTMEC4YR missing'
assert np.allclose(df['WTMEC4YR'], df['WTMEC2YR'] / 2.0, equal_nan=True), \
    'stored WTMEC4YR does not match WTMEC2YR / 2 -- the file is stale'
df['Age'] = df['RIDAGEYR']

# PHQ-9 total: all nine items required
dpq_cols = [f'DPQ{i:03d}' for i in [10, 20, 30, 40, 50, 60, 70, 80, 90]]
df['PHQ9'] = df[dpq_cols].replace({7: np.nan, 9: np.nan}).sum(axis=1, min_count=9)

# Cognition composite (60+): memory / fluency / speed, raw zeros -> missing
elig_60 = df['Age'] >= 60
for src, dst in [('memory_score',  'MEM_Z'),
                 ('fluency_score', 'FLUEN_Z'),
                 ('speed_score',   'SPEED_Z')]:
    raw = df[src].replace(0, np.nan)
    s   = raw.loc[elig_60].dropna()
    df[dst] = np.where(elig_60, (raw - s.mean()) / s.std(), np.nan)
df['COG_Z'] = df[['MEM_Z', 'FLUEN_Z', 'SPEED_Z']].mean(axis=1, skipna=False)

COVS = ['Age', 'RIAGENDR', 'BMXBMI', 'DMDEDUC2', 'Wear_Time', 'SLD010H']
MEDS = ['KDM_BA_advance', 'SII_log']

def age_stratum(a):
    if pd.isna(a) or a < 18: return None
    if a < 40: return '18-39'
    if a < 60: return '40-59'
    if a < 70: return '60-69'
    return '70+'
df['Age_stratum'] = df['Age'].apply(age_stratum)
strata = ['18-39', '40-59', '60-69', '70+']

print(f'Loaded 8_nhanes_analytic.csv: {df.shape}')

## Step 2 — Derive the full sample hierarchy

Every analysis set used in the manuscript is derived from the analytic file
and cross-checked against the sample sizes in the Mplus output files
(`m_dep_young_iv.out`, `m_dep_old_iv.out`, `m_cog_old_iv.out`, and the pooled
`M1_depression.out`).

In [ ]:
adults = df[df['Age'] >= 18].copy()
elder  = df[df['Age'] >= 60].copy()

# Shared upstream chain.
n_full    = len(df)
n_adults  = len(adults)
n_iv      = int(adults['IV'].notna().sum())   # IV and IS share the hourly profile
n_ra      = int(adults['RA'].notna().sum())   # RA needs minute-level M10/L5
n_both    = int(adults.dropna(subset=['IV'] + MEDS).shape[0])

# x-variable complete set (exposure + covariates + weight).
base_dep  = adults.dropna(subset=['IV', 'WTMEC4YR'] + COVS)
n_xcov    = len(base_dep)

# Per-covariate attrition, applied in the order of COVS so the list cannot
# drift from the model specification.
COV_LABEL = {'Age': 'age', 'RIAGENDR': 'sex', 'BMXBMI': 'body mass index',
             'DMDEDUC2': 'education', 'Wear_Time': 'accelerometer wear time',
             'SLD010H': 'sleep duration'}
assert set(COVS) == set(COV_LABEL), 'COV_LABEL has drifted from COVS'
cov_loss, _step = {}, adults.dropna(subset=['IV', 'WTMEC4YR'])
for _c in COVS:
    _before = len(_step)
    _step = _step.dropna(subset=[_c])
    cov_loss[_c] = _before - len(_step)
assert len(_step) == n_xcov, (
    f'covariate cascade ends at {len(_step):,} but the x-complete set is '
    f'{n_xcov:,} -- the two are built differently')

# Primary SEM analysis sets.
# Mplus retains rows with complete x-variables and drops only those missing on
# every dependent variable in the model.
n_drop_allY  = int(base_dep[MEDS + ['PHQ9']].isna().all(axis=1).sum())
n_sem_dep    = n_xcov - n_drop_allY

base_cog     = base_dep[base_dep['Age'] >= 60]
n_xcov_60    = len(base_cog)
# Mplus retains a 60+ case when the exogenous variables are complete and at
# least one dependent variable is observed; the dependent variables in the
# cognition model are BA_ADV, SII_LOG and the three cognitive indicators.
_COG_IND = ['MEM_Z', 'FLUEN_Z', 'SPEED_Z']
_COG_DEPS = [c for c in ['KDM_BA_advance', 'SII_log'] + _COG_IND
             if c in base_cog.columns]
n_sem_cog    = int(base_cog[_COG_DEPS].notna().any(axis=1).sum())
n_sem_cog_cc = int(base_cog['COG_Z'].notna().sum())   # complete-case count (COG_Z observed)

# Per-outcome (equation-level) samples for the supportive WLS.
n_eq_kdm  = int(adults.dropna(subset=['IV', 'KDM_BA_advance', 'WTMEC4YR'] + COVS).shape[0])
n_eq_sii  = int(adults.dropna(subset=['IV', 'SII_log',        'WTMEC4YR'] + COVS).shape[0])
n_eq_phq  = int(adults.dropna(subset=['IV', 'PHQ9',           'WTMEC4YR'] + COVS).shape[0])
n_eq_cog  = int(elder.dropna(subset=['IV', 'COG_Z',           'WTMEC4YR'] + COVS).shape[0])

# Complete-case listwise sets.
n_cc_dep  = int(adults.dropna(subset=['IV', 'PHQ9',  'WTMEC4YR'] + MEDS + COVS).shape[0])
n_cc_cog  = int(elder.dropna(subset=['IV', 'COG_Z', 'WTMEC4YR'] + MEDS + COVS).shape[0])

n_elder   = len(elder)
n_iv_60   = int(elder['IV'].notna().sum())

# Cross-check against the n in the Mplus .out files (read, not transcribed).
import re as _re
def _mplus_n(path):
    m = _re.search(r'Number of observations\s+(\d+)', open(path).read())
    if not m:
        raise RuntimeError(f'Could not read "Number of observations" from {path}')
    return int(m.group(1))

_MPLUS_DIR = BASE / 'scripts' / '9_mplus_models'

# The primary depression models are age-stratified; their n are summed.
MPLUS_DEP_YOUNG = _mplus_n(_MPLUS_DIR / 'm_dep_young_iv.out')
MPLUS_DEP_OLD   = _mplus_n(_MPLUS_DIR / 'm_dep_old_iv.out')
MPLUS_DEP = MPLUS_DEP_YOUNG + MPLUS_DEP_OLD
MPLUS_COG = _mplus_n(_MPLUS_DIR / 'm_cog_old_iv.out')
print(f'Mplus n read from .out files: depression {MPLUS_DEP_YOUNG:,} (18-59) + '
      f'{MPLUS_DEP_OLD:,} (60+) = {MPLUS_DEP:,}; cognition {MPLUS_COG:,}')

# The pooled secondary model (M1_depression) must equal the stratified sum.
MPLUS_DEP_POOLED = _mplus_n(_MPLUS_DIR / 'M1_depression.out')
assert MPLUS_DEP_POOLED == MPLUS_DEP, (
    f'pooled depression model n {MPLUS_DEP_POOLED:,} != stratified sum '
    f'{MPLUS_DEP:,}; the age cells do not partition the analysis set')
print(f'Pooled secondary model (M1_depression) n = {MPLUS_DEP_POOLED:,}  ->  '
      f'matches the stratified sum\n')
print('Sample hierarchy')
print(f'  Full pooled dataset                       : {n_full:,}')
print(f'  Adults aged 18+                           : {n_adults:,}')
print(f'  + valid accelerometry (IV)                : {n_iv:,}')
print(f'  + both biological mediators               : {n_both:,}')
print(f'  + complete covariates (x-complete)        : {n_xcov:,}')
print(f'  - rows missing on every dependent variable: {n_drop_allY:,}')
print(f'  = PRIMARY SEM set, depression             : {n_sem_dep:,}')
print(f'  = PRIMARY SEM set, cognition (60+)        : {n_sem_cog:,}')
print()
print(f'  equation-level, KDM-BA                    : {n_eq_kdm:,}')
print(f'  equation-level, log SII                   : {n_eq_sii:,}')
print(f'  equation-level, PHQ-9                     : {n_eq_phq:,}')
print(f'  equation-level, cognition                 : {n_eq_cog:,}')
print(f'  complete-case listwise, depression        : {n_cc_dep:,}')
print(f'  complete-case listwise, cognition         : {n_cc_cog:,}')
print()
ok_dep = (n_sem_dep == MPLUS_DEP)
ok_cog = (n_sem_cog == MPLUS_COG)
print(f'  cross-check vs Mplus depression n = {MPLUS_DEP:,}  ->  '
      f'{"MATCH" if ok_dep else "MISMATCH"}')
print(f'  cross-check vs Mplus cognition  n = {MPLUS_COG:,}  ->  '
      f'{"MATCH" if ok_cog else "MISMATCH"}')
if not (ok_dep and ok_cog):
    print()
    print('  The derived hierarchy does not reproduce the Mplus n. Most likely')
    print('  cause: Mplus keeps a case when the exogenous variables are complete')
    print('  and ANY dependent variable is observed, so a derivation that requires')
    print('  a complete composite will fall short. Reconcile before using Figure 1.')
assert ok_dep and ok_cog, 'Derived SEM sample sizes do not match the Mplus output.'

## Step 3 — Shared matplotlib style

In [ ]:
mpl.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Helvetica', 'Arial', 'DejaVu Sans'],
    'font.size': 7, 'axes.titlesize': 7, 'axes.labelsize': 7,
    'xtick.labelsize': 6, 'ytick.labelsize': 6, 'legend.fontsize': 6,
    'axes.linewidth': 0.5, 'lines.linewidth': 0.9,
    'pdf.fonttype': 42, 'ps.fonttype': 42, 'savefig.dpi': 600,
    'mathtext.default': 'regular',
})
MM = 1 / 25.4


## Step 4 — Figure 1: conceptual framework and sample flow

Panel b terminates in the two primary structural equation model analysis sets.
The complete-case listwise subsets used for Table 1 and for the
mediator-adjusted direct effect are shown as annotations beneath each terminal
box.

In [ ]:
def add_node(ax, x, y, w, h, label, edge='#333333', text='#000000',
             fontsize=6.5, fontweight='bold'):
    box = FancyBboxPatch((x - w/2, y - h/2), w, h,
                          boxstyle='round,pad=0.02,rounding_size=0.18',
                          facecolor='white', edgecolor=edge, linewidth=1.5)
    ax.add_patch(box)
    ax.text(x, y, label, ha='center', va='center',
            fontsize=fontsize, fontweight=fontweight, color=text,
            linespacing=1.15)
    return (x, y, w, h)

def edge_pt(node, target):
    x, y, w, h = node
    tx, ty = target
    dx, dy = tx - x, ty - y
    if dx == 0 and dy == 0:
        return (x, y)
    s = min(w / 2 / max(abs(dx), 1e-6), h / 2 / max(abs(dy), 1e-6))
    return (x + s * dx, y + s * dy)

def add_arrow(ax, src, dst, color='black'):
    ax.add_patch(FancyArrowPatch(edge_pt(src, (dst[0], dst[1])),
                                  edge_pt(dst, (src[0], src[1])),
                                  arrowstyle='-|>', mutation_scale=14,
                                  linewidth=1.4, color=color,
                                  connectionstyle='arc3,rad=0.0'))

fig1 = plt.figure(figsize=(183 * MM, 95 * MM))
gs   = GridSpec(1, 2, figure=fig1, width_ratios=[1.0, 1.0], wspace=0.16)

# Panel a: conceptual framework.
axA = fig1.add_subplot(gs[0, 0])
axA.set_xlim(0, 10); axA.set_ylim(0, 10); axA.axis('off')
axA.set_title('A   Parallel-pathway framework',
               loc='left', fontweight='bold', pad=6)

# Node text is 6 pt on an ~88 mm panel.
iv  = add_node(axA, 1.45, 5.0, 2.7, 1.7,
                'Rest-activity\nrhythm\n(RA \u00b7 IS \u00b7 IV)',
                edge=IV_COLOR, text=IV_COLOR, fontsize=6)
ba  = add_node(axA, 4.95, 7.5, 3.0, 1.5,
                'KDM biological-age\nadvancement',
                edge=KDM_BA_COLOR, text=KDM_BA_COLOR, fontsize=6)
sii = add_node(axA, 4.95, 2.5, 3.0, 1.7,
                'log SII\n(systemic immune\u2013\ninflammation index)',
                edge=SII_COLOR, text=SII_COLOR, fontsize=6)
phq = add_node(axA, 8.5, 7.5, 2.9, 1.5,
                'Depressive\nsymptoms (PHQ-9)',
                edge=PHQ9_COLOR, text=PHQ9_COLOR, fontsize=6)
cog = add_node(axA, 8.5, 2.5, 2.9, 1.5,
                'Cognitive function\n(adults 60+)',
                edge=COG_COLOR, text=COG_COLOR, fontsize=6)

for s, d, c in [(iv, ba, KDM_BA_COLOR), (iv, sii, SII_COLOR),
                (ba, phq, KDM_BA_COLOR), (ba, cog, KDM_BA_COLOR),
                (sii, phq, SII_COLOR),   (sii, cog, SII_COLOR),
                (iv, phq, '#666666'),    (iv, cog, '#666666')]:
    add_arrow(axA, s, d, color=c)

axA.annotate('', xy=(5.0, 3.3), xytext=(5.0, 6.7),
              arrowprops=dict(arrowstyle='<|-|>', color='#aaaaaa',
                              linewidth=0.8, linestyle='--',
                              connectionstyle='arc3,rad=-0.45'))
axA.text(3.5, 5.0, r'$\Psi_{BA,SII}$', ha='center', va='center',
          fontsize=6, color='#777777', style='italic')
axA.text(0.01, 0.01,
          "a and b paths: coloured arrows;  direct path (c′): grey;  "
          'residual mediator covariance: dashed',
          transform=axA.transAxes, fontsize=5.5, color='#444444', ha='left')

# Panel b: sample flow.
axB = fig1.add_subplot(gs[0, 1])
axB.set_xlim(0, 10); axB.set_ylim(0, 10); axB.axis('off')
axB.set_title('B   Sample selection',
               loc='left', fontweight='bold', pad=6)

chain = [
    (5.0, 9.3, f'NHANES 2011-2014 pooled\nn = {n_full:,}',                        GREY),
    (5.0, 7.7, f'Adults aged ≥ 18 years\nn = {n_adults:,}',                   IV_GRADIENT_4[0]),
    (5.0, 6.1, f'+ valid wrist accelerometry\nn = {n_iv:,}',                       IV_GRADIENT_4[1]),
    (5.0, 4.5, f'+ complete covariates\nn = {n_xcov:,}',                           IV_GRADIENT_4[2]),
]
for (x, y, lab, color) in chain:
    axB.add_patch(FancyBboxPatch((x - 2.0, y - 0.62), 4.0, 1.24,
                                  boxstyle='round,pad=0.04,rounding_size=0.12',
                                  edgecolor='black', linewidth=0.9,
                                  facecolor=color, alpha=0.32))
    axB.text(x, y, lab, ha='center', va='center', fontsize=6, linespacing=1.2)

terminals = [
    (2.4, 2.2, f'Depression\nanalysis set\nn = {n_sem_dep:,}',       PHQ9_COLOR),
    (7.6, 2.2, f'Cognition (60+)\nanalysis set\nn = {n_sem_cog:,}',  COG_COLOR),
]
for (x, y, lab, color) in terminals:
    axB.add_patch(FancyBboxPatch((x - 1.85, y - 0.80), 3.7, 1.60,
                                  boxstyle='round,pad=0.04,rounding_size=0.12',
                                  edgecolor='black', linewidth=1.4,
                                  facecolor=color, alpha=0.34))
    axB.text(x, y, lab, ha='center', va='center', fontsize=6,
             fontweight='bold', linespacing=1.25)

for x1, y1, x2, y2 in [(5, 8.68, 5, 8.32), (5, 7.08, 5, 6.72),
                        (5, 5.48, 5, 5.12),
                        (5, 3.88, 2.4, 3.00), (5, 3.88, 7.6, 3.00)]:
    axB.annotate('', xy=(x2, y2), xytext=(x1, y1),
                  arrowprops=dict(arrowstyle='->', color='black', lw=0.9))

axB.text(2.4, 1.05,
          f'complete-case subset\nn = {n_cc_dep:,}',
          ha='center', va='center', fontsize=5.5, color='#555555',
          style='italic', linespacing=1.2)
axB.text(7.6, 1.05,
          f'complete-case subset\nn = {n_cc_cog:,}',
          ha='center', va='center', fontsize=5.5, color='#555555',
          style='italic', linespacing=1.2)
axB.text(0.1, -0.02,
          'Bold boxes: primary SEM analysis sets. Counts follow the 'f'IV/IS sample;\nRA samples differ by {n_iv - n_ra} at the'
          'accelerometry step. \nEquation-level samples: Supp Table S12.',
          transform=axB.transAxes, fontsize=5.5, color='#444444',
          ha='left', va='bottom', linespacing=1.3)

fig1.subplots_adjust(left=0.03, right=0.98, top=0.92, bottom=0.06)
# No bbox_inches='tight': margins are set by subplots_adjust above, and tight
# cropping expands the canvas past 183 mm when out-of-axes text sticks out.
fig1.savefig(F_MAIN / 'Fig1_Study_Overview.pdf')
fig1.savefig(F_MAIN / 'Fig1_Study_Overview.png', dpi=600)
plt.close(fig1)
print(f'wrote {F_MAIN / "Fig1_Study_Overview.pdf"} + .png')

# Source data for panel b.
_src = pd.DataFrame([
    ('selection chain', 'NHANES 2011-2014 pooled', n_full),
    ('selection chain', 'adults >= 18 y', n_adults),
    ('selection chain', 'valid wrist accelerometry (IV/IS)', n_iv),
    ('selection chain', 'valid wrist accelerometry (RA)', n_ra),
    ('selection chain', 'complete covariates', n_xcov),
    ('terminal', 'depression SEM analysis set', n_sem_dep),
    ('terminal', 'cognition SEM analysis set (60+)', n_sem_cog),
    ('annotation', 'depression complete-case subset', n_cc_dep),
    ('annotation', 'cognition complete-case subset', n_cc_cog),
    ('cross-check', 'Mplus n, depression 18-59 (m_dep_young_iv.out)', MPLUS_DEP_YOUNG),
    ('cross-check', 'Mplus n, depression 60+ (m_dep_old_iv.out)', MPLUS_DEP_OLD),
    ('cross-check', 'Mplus n, depression, both strata', MPLUS_DEP),
    ('cross-check', 'Mplus n, cognition 60+ (m_cog_old_iv.out)', MPLUS_COG),
], columns=['group', 'quantity', 'n'])
with pd.ExcelWriter(F_SRC / 'Source_Data_Fig1_Study_Overview.xlsx',
                    engine='openpyxl') as xw:
    _src.to_excel(xw, sheet_name='source data', index=False)
    pd.DataFrame({'Notes': [
        'Fig. 1 | Conceptual framework and sample selection.',
        'Panel a is schematic and has no source data; panel b counts are '
        'derived from the analytic file and cross-checked against the n '
        'reported in the Mplus output files.',
        'Counts follow the IV/IS sample; the relative-amplitude sample '
        'differs at the accelerometry step (see rows above).',
    ]}).to_excel(xw, sheet_name='legend', index=False)
print(f'wrote {F_SRC / "Source_Data_Fig1_Study_Overview.xlsx"}')

for _stale in [F_MAIN / 'Fig1_Study_Overview.tiff']:
    if _stale.exists():
        _dest = BASE / '_superseded' / _stale.name
        _dest.parent.mkdir(exist_ok=True)
        _stale.rename(_dest)
        print(f'superseded -> _superseded/{_stale.name}')

## Step 5 — Supplementary sample-accounting table

Enumerates every analysis set used anywhere in the manuscript, with the
inclusion rule and the tables and figures in which each appears.

In [ ]:
# Variable roles: Mplus treats predictors as fixed, so a case missing any X is
# dropped, whereas a case missing some Y is retained under full-information
# maximum likelihood.
_X_ROLE = ('Rhythm metric (RA, IS or IV); age; sex; body mass index; '
           'education; accelerometer wear time; sleep duration; survey '
           'weight, stratum and primary sampling unit')
_Y_DEP = 'KDM-BA advancement; log SII; PHQ-9 total'
_Y_COG = 'KDM-BA advancement; log SII; memory, fluency and speed z-scores'

# The interaction models' n, derived here and checked against x_age_iv.out.
n_interaction = int(base_dep['KDM_BA_advance'].notna().sum())
MPLUS_INTER = _mplus_n(_MPLUS_DIR / 'x_age_iv.out')
assert n_interaction == MPLUS_INTER, (
    f'derived interaction n {n_interaction:,} != Mplus {MPLUS_INTER:,}')

# Internal reconciliation: the chain has to close arithmetically.
assert n_iv - sum(cov_loss.values()) == n_xcov
assert n_xcov - n_drop_allY == n_sem_dep
assert n_sem_dep == len(base_dep) - n_drop_allY
print(f'sample chain reconciles: {n_iv:,} - {sum(cov_loss.values())} covariate '
      f'losses = {n_xcov:,}; - {n_drop_allY} with no observed dependent = '
      f'{n_sem_dep:,}; interaction n {n_interaction:,} matches x_age_iv.out')

rows = [
    ('Variable role', 'Must be observed (exogenous, X)', _X_ROLE, n_xcov,
     'Methods, Missing data'),
    ('Variable role', 'May be missing (endogenous, Y) - depression models',
     _Y_DEP, n_sem_dep, 'Methods, Missing data'),
    ('Variable role', 'May be missing (endogenous, Y) - cognition models',
     _Y_COG, n_sem_cog, 'Methods, Missing data'),

    ('Selection chain', 'Pooled NHANES 2011-2012 and 2013-2014',
     'All participants', n_full, 'Figure 1b'),
    ('Selection chain', 'Adults aged 18 years and older',
     'Age >= 18 y', n_adults, 'Figure 1b'),
    ('Selection chain', 'Valid wrist accelerometry (IV/IS)',
     '+ hourly profile derivable (>= 10 h/day wear on >= 4 days)', n_iv, 'Figure 1b'),
    ('Selection chain', 'Valid wrist accelerometry (RA)',
     '+ minute-level M10/L5 derivable', n_ra, 'Figure 1b note'),
    ('Selection chain', 'Both biological mediators observed',
     '+ KDM-BA advancement and log SII non-missing', n_both, '-'),
    ('Selection chain', 'Complete covariates (x-complete)',
     'add age, sex, BMI, education, wear time, sleep duration, survey weight',
     n_xcov, 'Figure 1b'),
] + [
    ('Covariate attrition', f'Lost to missing {COV_LABEL[c]}',
     f'Applied in order to the {n_iv:,} adults with valid accelerometry',
     cov_loss[c], '-')
    for c in COVS if cov_loss[c] > 0
] + [
    ('Selection chain', 'Adults aged 60 years and older',
     'Age >= 60 y', n_elder, '-'),
    ('Selection chain', 'Adults 60+ with valid accelerometry',
     'Age >= 60 y and IV derivable', n_iv_60, '-'),

    ('Primary SEM', 'Depression structural equation model',
     'x-complete, excluding rows missing on every dependent variable '
     f'(n = {n_drop_allY} excluded)', n_sem_dep, 'Table 2; Fig. 4'),
    ('Primary SEM', 'Cognition structural equation model (60+)',
     'x-complete and aged 60+, excluding rows missing on every dependent '
     'variable of that model, i.e. on all of KDM-BA advancement, log SII and '
     f'the three cognition indicators (n = {n_xcov_60 - n_sem_cog} excluded '
     f'of {n_xcov_60:,} x-complete adults aged 60+). This is larger than the '
     f'complete-case subset with the cognition composite observed ({n_cc_cog:,}).',
     n_sem_cog, 'Table 2; Fig. 4'),
    ('Primary SEM', 'Age-interaction models',
     'x-complete with KDM-BA advancement observed; it is the only dependent '
     'variable in these models, so full-information estimation recovers '
     'nothing here', n_interaction, 'Table 3'),
    ('FIML recovery', 'Retained by FIML but lost to listwise deletion',
     'Depression models: analysis sample minus the complete-case subset',
     n_sem_dep - n_cc_dep, 'Methods, Missing data'),

    ('Supportive WLS', 'IV to KDM-BA advancement equation',
     'IV + KDM-BA advancement + covariates', n_eq_kdm, 'Supp. Table S14 (a1 path)'),
    ('Supportive WLS', 'IV to log SII equation',
     'IV + log SII + covariates', n_eq_sii, 'Supp. Table S14 (a2 path)'),
    ('Supportive WLS', 'IV to PHQ-9 equation',
     'IV + PHQ-9 + covariates', n_eq_phq, 'Supp. Table S15 (M1, M2)'),
    ('Supportive WLS', 'IV to cognition composite equation (60+)',
     'IV + cognition composite + covariates, aged 60+', n_eq_cog,
     'Supp. Table S15 (M1, M2)'),

    ('Complete-case', 'Depression, listwise on all model variables',
     'IV + both mediators + PHQ-9 + covariates', n_cc_dep,
     'Table 1; Supp. Table S15 (M3)'),
    ('Complete-case', 'Cognition, listwise on all model variables (60+)',
     'IV + both mediators + cognition composite + covariates, aged 60+',
     n_cc_cog, 'Supp. Table S15 (M3)'),
]

accounting = pd.DataFrame(
    rows, columns=['Category', 'Analysis set', 'Inclusion rule', 'n',
                   'Reported in'])
accounting['n'] = accounting['n'].map('{:,}'.format)
print(accounting.to_string(index=False))

accounting.to_csv(R_SUPP / 'SuppTableS12_Sample_Accounting.csv', index=False)
try:
    with pd.ExcelWriter(R_SUPP / 'SuppTableS12_Sample_Accounting.xlsx',
                        engine='openpyxl') as xw:
        accounting.to_excel(xw, sheet_name='Supp Table S12', index=False)
        pd.DataFrame({'Notes': [
            'Three estimation strategies are used and legitimately yield different sample sizes.',
            'Primary SEM (Mplus TYPE = COMPLEX, MLR): retains rows with complete exposure and covariates, '
            'applying full-information maximum likelihood to partially observed mediator and outcome variables.',
            'Supportive weighted least squares: uses the equation-level sample for each structural equation, '
            'so that each supportive estimate is directly comparable to the corresponding Mplus STDYX coefficient.',
            'Complete-case listwise: requires every variable in the joint model; used for descriptive '
            'comparability (Table 1) and for the mediator-adjusted direct effect (Table 3, M3).',
            'The cognition composite is the mean of the within-60+ z-scores of the memory, fluency, and '
            'processing-speed domain scores, with raw zeros recoded to missing.',
            'In the structural equation model the cognition outcome is a latent factor with three observed '
            'indicators; in the supportive regressions it is the observed composite score.',
            f'Derived sample sizes were cross-checked against the Mplus output: depression n = {n_sem_dep:,}, '
            f'cognition n = {n_sem_cog:,}.',
        ]}).to_excel(xw, sheet_name='Notes', index=False)
    print(f'\nWrote {R_SUPP / "SuppTableS12_Sample_Accounting.xlsx"}')
except Exception as e:
    print(f'(xlsx skipped: {e})')
print(f'Wrote {R_SUPP / "SuppTableS12_Sample_Accounting.csv"}')

# Fig 1's source data is written in the figure cell
# (Source_Data_Fig1_Study_Overview.xlsx); move the workbook under the older
# name to _superseded/.
_old_src = F_SRC / 'Source_Data_Fig1.xlsx'
if _old_src.exists():
    _dest = BASE / '_superseded' / _old_src.name
    _dest.parent.mkdir(exist_ok=True)
    _old_src.rename(_dest)
    print(f'superseded -> _superseded/{_old_src.name}')
for _stale in [R_SUPP / 'SuppTable_Sample_Accounting.csv',
               R_SUPP / 'SuppTable_Sample_Accounting.xlsx']:
    if _stale.exists():
        _dest = BASE / '_superseded' / _stale.name
        _dest.parent.mkdir(exist_ok=True)
        _stale.rename(_dest)
        print(f'superseded -> _superseded/{_stale.name}')


## Step 6 — Supp Fig S4: exposure and mediator distributions by age

The complete-case depression sample is used so that all panels describe one
fixed set of participants.

In [ ]:
# Distributions of the three rhythm metrics and the two mediators, by age
# stratum, on the complete-case depression sample.
dist_sample = adults.dropna(subset=['IV', 'PHQ9', 'WTMEC4YR'] + MEDS + COVS).copy()
print(f'Distribution figure sample (complete-case depression): n = {len(dist_sample):,}')

fig2 = plt.figure(figsize=(183 * MM, 105 * MM))
gs2 = GridSpec(2, 6, figure=fig2, hspace=0.55, wspace=0.9)
strata_colors = IV_GRADIENT_4

# Top row: one histogram per rhythm metric.
for j, exp in enumerate(EXP_ORDER):
    ax = fig2.add_subplot(gs2[0, 2*j:2*j+2])
    v = dist_sample[exp].dropna()
    lo_q, hi_q = v.quantile([0.005, 0.995])
    for s, c in zip(strata, strata_colors):
        sub = dist_sample.loc[dist_sample['Age_stratum'] == s, exp].dropna()
        if len(sub) > 5:
            ax.hist(sub, bins=35, range=(lo_q, hi_q), density=True,
                    alpha=0.45, color=c, edgecolor='black', linewidth=0.2,
                    label=f'{s} (n = {len(sub):,})')
    ax.set_xlim(lo_q, hi_q)
    ax.set_xlabel(f'{EXP_NAME[exp]} ({exp})')
    if j == 0:
        ax.set_ylabel('Density')
    ax.set_title(chr(ord("a") + j), loc='left', fontweight='bold')
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
    if j == 2:
        ax.legend(loc='upper left', frameon=False, fontsize=5,
                  title='Age stratum', title_fontsize=5.5)

# Bottom row: the two mediators as box plots across strata.
for j, (col, ylab, ttl, colr) in enumerate([
        ('KDM_BA_advance', 'KDM-BA advancement (years)', 'd', KDM_BA_COLOR),
        ('SII_log', 'log SII', 'e', SII_COLOR)]):
    ax = fig2.add_subplot(gs2[1, 3*j:3*j+3])
    bp_data = [dist_sample.loc[dist_sample['Age_stratum'] == s, col].dropna()
               for s in strata]
    bp = ax.boxplot(bp_data, tick_labels=strata, showfliers=False,
                    patch_artist=True, widths=0.55,
                    medianprops=dict(color='black', lw=1.0),
                    whiskerprops=dict(lw=0.7), capprops=dict(lw=0.7))
    for patch, c in zip(bp['boxes'], strata_colors):
        patch.set_facecolor(c); patch.set_alpha(0.55); patch.set_linewidth(0.6)
    ax.set_ylabel(ylab)
    ax.set_xlabel('Age stratum')
    ax.set_title(ttl, loc='left', fontweight='bold', color=colr)
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

stem = 'SuppFigS4_Exposure_Mediator_Distributions'
fig2.savefig(F_SUPP / f'{stem}.pdf', bbox_inches='tight')
fig2.savefig(F_SUPP / f'{stem}.png', bbox_inches='tight', dpi=600)
plt.close(fig2)
print(f'wrote {F_SUPP / stem}.pdf + .png')

# Source data: per-stratum summaries for every panel.
rows = []
for var in list(EXP_ORDER) + ['KDM_BA_advance', 'SII_log']:
    for s in strata:
        v = dist_sample.loc[dist_sample['Age_stratum'] == s, var].dropna()
        rows.append(dict(variable=var, stratum=s, n=len(v),
                         mean=v.mean(), sd=v.std(), median=v.median(),
                         q1=v.quantile(.25), q3=v.quantile(.75),
                         p5=v.quantile(.05), p95=v.quantile(.95)))
cap = ['Supplementary Fig. S4 | Distributions of the rhythm metrics and '
       'mediators by age stratum.',
       'Complete-case depression sample. Histograms truncated at the 0.5th '
       'and 99.5th percentiles for display; boxes span the interquartile '
       'range, whiskers 1.5 IQR, outliers not drawn.',
       'Panel-level summaries in the source-data sheet are computed on the '
       'full untruncated values.']
with pd.ExcelWriter(F_SRC / f'Source_Data_{stem}.xlsx', engine='openpyxl') as xw:
    pd.DataFrame(rows).to_excel(xw, sheet_name='source data', index=False)
    pd.DataFrame({'Notes': cap}).to_excel(xw, sheet_name='legend', index=False)
print(f'wrote {F_SRC / ("Source_Data_" + stem + ".xlsx")}')

for _stale in [F_SUPP / 'SuppFig_Exposure_Mediator_Distributions.png',
               F_SUPP / 'SuppFig_Exposure_Mediator_Distributions.tiff',
               F_SRC / 'Source_Data_SuppFig_Distributions.xlsx']:
    if _stale.exists():
        _dest = BASE / '_superseded' / _stale.name
        _dest.parent.mkdir(exist_ok=True)
        _stale.rename(_dest)
        print(f'superseded -> _superseded/{_stale.name}')
